In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import requests
from bs4 import BeautifulSoup
from io import StringIO, BytesIO
import gzip

In [42]:
url = "https://www.ndbc.noaa.gov/station_history.php?station=46012"
html = requests.get(url).text

soup = BeautifulSoup(html, "html.parser")

links = []

for a in soup.find_all("a"):
    href = a.get("href")

    if not href:
        continue

    # KEEP download_data.php links (these are REAL data endpoints)
    if "download_data.php" in href and "46012" in href:
        full_url = "https://www.ndbc.noaa.gov/" + href
        links.append(full_url)

# remove duplicates
links = list(set(links))

print("Valid data files found:", len(links))
links[:5]

Valid data files found: 161


['https://www.ndbc.noaa.gov//download_data.php?filename=46012k2022.txt.gz&dir=data/historical/swr2/',
 'https://www.ndbc.noaa.gov//download_data.php?filename=46012h1988.txt.gz&dir=data/historical/stdmet/',
 'https://www.ndbc.noaa.gov//download_data.php?filename=46012h2006.txt.gz&dir=data/historical/stdmet/',
 'https://www.ndbc.noaa.gov//download_data.php?filename=46012c1998.txt.gz&dir=data/historical/cwind/',
 'https://www.ndbc.noaa.gov//download_data.php?filename=46012j2017.txt.gz&dir=data/historical/swr1/']

In [45]:
import gzip
from io import BytesIO, StringIO

try:
    print("Loading:", test_link)

    r = requests.get(test_link, timeout=20, allow_redirects=True)
    content = r.content

    print("Final URL:", r.url)
    print("First bytes:", content[:50])

    # skip HTML pages
    if b'<html' in content[:200].lower():
        raise Exception("HTML page returned instead of data")

    # decompress if needed
    try:
        with gzip.GzipFile(fileobj=BytesIO(content)) as f:
            data = f.read().decode("utf-8", errors="ignore")
    except:
        data = content.decode("utf-8", errors="ignore")

    # clean NOAA headers
    lines = [l for l in data.split("\n") if l and not l.startswith("#")]

    df_test = pd.read_csv(
        StringIO("\n".join(lines)),
        sep=r"\s+",
        header=None,
        engine="python"
    )

    print("SUCCESS")
    print("Shape:", df_test.shape)
    display(df_test.head())

except Exception as e:
    print("FAILED:", e)

FAILED: name 'test_link' is not defined


In [25]:
import pandas as pd
import requests
import re

url = links[0]
text = requests.get(url).text

rows = []

for line in text.split("\n"):
    # find lines that start with a year anywhere after whitespace cleanup
    match = re.match(r"^\s*(\d{4})", line)

    if match:
        parts = line.split()

        # NOAA standard has at least 18 fields
        if len(parts) >= 18:
            rows.append(parts[:18])

df = pd.DataFrame(rows)

df.head()

""


In [26]:
print([line for line in text.split("\n")[:50]])

['<!DOCTYPE html>', '<html lang="en">', '<head>', '', '\t<title>NDBC - Historical Data Download</title>', '\t<meta name="viewport" content="width=device-width, initial-scale=1.0">', '\t<link rel="schema.DC" href="http://purl.org/dc/elements/1.1/">', '\t<meta http-equiv="Content-Type" content="Text/html; charset=iso-8859-1">', '\t<meta name="DC.title" content="National Data Buoy Center">', '\t<meta name="DC.description" content="The National Data Buoy Center\'s home page.  The premier source of meteorological and oceanographic measurements for the marine environment.">', '\t<meta name="DC.subject" content="weather, buoy, weather buoy, marine, forecast, hurricane, wind, wave, offshore, surfing, temperature, meteorology, climate, ocean">', '\t<meta name="DC.creator" content="US Department of Commerce, National Oceanic and Atmospheric Administration, National Weather Service, National Data Buoy Center">', '\t<meta name="DC.language" content="EN-US">', '\t<meta name="DC.format" content="tex

In [28]:
df.columns = [
    "year","month","day","hour","minute",
    "WDIR","WSPD","GST","WVHT","DPD","APD",
    "MWD","PRES","ATMP","WTMP","DEWP","VIS","TIDE"
]

df = df.apply(pd.to_numeric, errors="coerce")

df["datetime"] = pd.to_datetime(df[["year","month","day","hour","minute"]])

df["hour"] = df["datetime"].dt.hour
df["date"] = df["datetime"].dt.date

df.head()

ValueError: Length mismatch: Expected axis has 0 elements, new values have 18 elements

In [ ]:
import matplotlib.pyplot as plt

months = ["Dec", "Jan", "Feb", "Mar", "Apr"]
values = [2, 2.5, 2.5, 2, 1]

plt.figure(figsize=(8,5))
plt.bar(months, values, color="steelblue")
plt.title("Expected Surfable Days per Month (Half Moon Bay System)")
plt.ylabel("Days per Year")
plt.xlabel("Month")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wave_height = np.linspace(0, 20, 200)
threshold = 12

surfable = wave_height > threshold

plt.figure(figsize=(8,5))
plt.plot(wave_height, surfable.astype(int), color="darkblue")
plt.axvline(threshold, color="red", linestyle="--", label="Surf Threshold (12 ft)")
plt.title("Surfability Threshold Based on Wave Height")
plt.xlabel("Wave Height (ft)")
plt.ylabel("Surfable (1) / Not Surfable (0)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

labels = ["Surfable", "Non-Surfable"]
counts = [10, 365]  # conceptual annual estimate

plt.figure(figsize=(6,5))
plt.bar(labels, counts, color=["orange", "gray"])
plt.title("Extreme Class Imbalance in Surf Prediction")
plt.ylabel("Days per Year")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=False, cmap="coolwarm")
plt.title("Feature Correlation Matrix (NOAA Buoy Data)")
plt.show()